# SciFact: Contriever Retrieval

Runs **two modes** on SciFact using `facebook/contriever` as the embedding model:

| Mode | Query vector |
|---|---|
| **Contriever-Baseline** | encode original query |
| **Contriever-HyDE** | encode GPT-4o-mini hypothetical document (same generator as `run_hyde.ipynb`) |

Contriever uses **mean pooling** over all token embeddings (not CLS).  
Corpus embeddings are L2-normalised before FAISS inner-product search, giving cosine similarity.

In [24]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

try:
    import torch
    torch.set_num_threads(1)
    torch.set_num_interop_threads(1)
except Exception:
    pass

print('Thread guards enabled.')

Thread guards enabled.


In [25]:
from getpass import getpass

if not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('Enter your OpenAI API key: ').strip()

print('OPENAI_API_KEY is set:', bool(os.getenv('OPENAI_API_KEY')))

OPENAI_API_KEY is set: True


In [26]:
import json
from pathlib import Path

import faiss
import numpy as np
from tqdm import tqdm

from pyserini.encode import AutoQueryEncoder

from config import DATASET_SPLIT, PREFER_BEIR, MAX_QUERIES, TOP_KS
from load_data import load_scifact_data
from evaluate import evaluate_run
from generate_hyde import HyDEGenerator

print('Imports OK')

Imports OK


## Contriever Encoder

Uses Pyserini's `AutoQueryEncoder` — the same interface as `hyde/hyde-dl19.ipynb`.  
`pooling='mean'` applies mean pooling over all non-padding tokens, matching Contriever's original design.

In [27]:
CONTRIEVER_MODEL = 'facebook/contriever'
BATCH_SIZE = 64

# Identical constructor to hyde/hyde-dl19.ipynb
query_encoder = AutoQueryEncoder(
    encoder_dir=CONTRIEVER_MODEL,
    pooling='mean',
    use_fp16=False,
)

print(f'Loaded {CONTRIEVER_MODEL} via Pyserini AutoQueryEncoder')


def encode(texts: list, normalize: bool = True) -> np.ndarray:
    """Encode a list of texts with Contriever; optionally L2-normalise for cosine similarity."""
    all_vecs = []
    for text in texts:
        vec = query_encoder.encode(text)   # same call as in hyde-dl19.ipynb
        all_vecs.append(np.array(vec))
    vecs = np.vstack(all_vecs).astype(np.float32)
    if normalize:
        norms = np.linalg.norm(vecs, axis=1, keepdims=True)
        vecs = vecs / np.clip(norms, 1e-9, None)
    return vecs

Loaded facebook/contriever via Pyserini AutoQueryEncoder


## Load SciFact Data

In [28]:
SPLIT             = DATASET_SPLIT
PREFER_BEIR_LOCAL = PREFER_BEIR
MAX_QUERIES_LOCAL = MAX_QUERIES
TOP_KS_LOCAL      = [1, 10, 100]   # Recall@5 replaced by Recall@100
MAX_K             = max(TOP_KS_LOCAL)

corpus, queries, qrels, data_source = load_scifact_data(
    split=SPLIT,
    prefer_beir=PREFER_BEIR_LOCAL,
    max_queries=MAX_QUERIES_LOCAL,
)

query_ids      = list(queries.keys())
corpus_doc_ids = list(corpus.keys())
corpus_texts   = [corpus[did] for did in corpus_doc_ids]

print(f'Data source : {data_source}')
print(f'Corpus size : {len(corpus_texts)}')
print(f'Queries     : {len(query_ids)}')

Data source : mteb/scifact (corpus:corpus/corpus, queries:queries/queries, qrels:default/test)
Corpus size : 5183
Queries     : 300


## Encode Corpus & Build FAISS Index

The corpus embeddings are shared between both retrieval modes — encode once, reuse twice.

In [29]:
print('Encoding corpus with Contriever...')
corpus_vectors = encode(corpus_texts)          # (N, 768), L2-normalised
print(f'Corpus vectors shape: {corpus_vectors.shape}')

dim   = corpus_vectors.shape[1]
index = faiss.IndexFlatIP(dim)                 # inner product == cosine after normalisation
index.add(corpus_vectors)
print(f'FAISS index built with {index.ntotal} vectors')

Encoding corpus with Contriever...
Corpus vectors shape: (5183, 768)
FAISS index built with 5183 vectors


## Mode 1 — Contriever Baseline

Encode each query directly and retrieve top-K passages.

In [30]:
query_texts   = [queries[qid] for qid in query_ids]
query_vectors = encode(query_texts)

_, all_indices = index.search(query_vectors, MAX_K)

baseline_retrieved = {}
baseline_rows      = []
for i, qid in enumerate(query_ids):
    doc_ids = [corpus_doc_ids[j] for j in all_indices[i].tolist()]
    baseline_retrieved[qid] = doc_ids
    gold = qrels[qid]
    baseline_rows.append({
        'mode': 'contriever_baseline',
        'query_id': qid,
        'query': queries[qid],
        'gold_doc_ids': sorted(gold),
        'retrieved_doc_ids': doc_ids,
        'hit@1':   int(any(d in gold for d in doc_ids[:1])),
        'hit@10':  int(any(d in gold for d in doc_ids[:10])),
        'hit@100': int(any(d in gold for d in doc_ids[:100])),
    })

baseline_metrics = evaluate_run(baseline_retrieved, qrels, TOP_KS_LOCAL)

print('=== Contriever Baseline ===')
for k in ['Recall@1', 'Recall@10', 'Recall@100', 'MRR@10', 'nDCG@10']:
    if k in baseline_metrics:
        print(f'{k:<12}: {baseline_metrics[k]:.4f}')

=== Contriever Baseline ===
Recall@1    : 0.4183
Recall@10   : 0.7353
Recall@100  : 0.9027
MRR@10      : 0.5453
nDCG@10     : 0.5847


In [31]:
out_baseline = Path('results/contriever_baseline')
out_baseline.mkdir(parents=True, exist_ok=True)

payload = {
    'config': {
        'mode': 'contriever_baseline',
        'embed_model': CONTRIEVER_MODEL,
        'split': SPLIT,
        'data_source': data_source,
        'top_ks': TOP_KS_LOCAL,
        'max_queries': MAX_QUERIES_LOCAL,
    },
    'contriever_baseline': baseline_metrics,
}
with (out_baseline / 'metrics.json').open('w') as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)
with (out_baseline / 'per_query_results.jsonl').open('w') as f:
    for row in baseline_rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f'Saved to {out_baseline}')

Saved to results/contriever_baseline


## Mode 2 — Contriever + HyDE

For each query:
1. GPT-4o-mini generates one hypothetical scientific passage (same prompt as `run_hyde.ipynb`)
2. Encode the hypothetical passage with Contriever
3. Retrieve top-K passages using the hypothetical-doc vector

In [32]:
hyde_generator = HyDEGenerator(model_name='gpt-4o-mini', provider='openai')
print('HyDE generator:', hyde_generator.model_name)

hyde_docs = {}
for qid in tqdm(query_ids, desc='Generate HyDE docs'):
    hyde_docs[qid] = hyde_generator.generate(queries[qid])

print(f'Generated {len(hyde_docs)} hypothetical documents')

HyDE generator: gpt-4o-mini


Generate HyDE docs: 100%|██████████| 300/300 [16:10<00:00,  3.24s/it]

Generated 300 hypothetical documents


In [36]:
hyde_texts   = [hyde_docs[qid] for qid in query_ids]
hyde_vectors = encode(hyde_texts)

_, all_indices_hyde = index.search(hyde_vectors, MAX_K)

hyde_retrieved = {}
hyde_rows      = []
for i, qid in enumerate(query_ids):
    doc_ids = [corpus_doc_ids[j] for j in all_indices_hyde[i].tolist()]
    hyde_retrieved[qid] = doc_ids
    gold = qrels[qid]
    hyde_rows.append({
        'mode': 'contriever_hyde',
        'query_id': qid,
        'query': queries[qid],
        'hyde_document': hyde_docs[qid],
        'gold_doc_ids': sorted(gold),
        'retrieved_doc_ids': doc_ids,
        'hit@1':   int(any(d in gold for d in doc_ids[:1])),
        'hit@10':  int(any(d in gold for d in doc_ids[:10])),
        'hit@100': int(any(d in gold for d in doc_ids[:100])),
    })

hyde_metrics = evaluate_run(hyde_retrieved, qrels, TOP_KS_LOCAL)

print('=== Contriever + HyDE ===')
for k in ['Recall@1', 'Recall@10', 'Recall@100', 'MRR@10', 'nDCG@10']:
    if k in hyde_metrics:
        print(f'{k:<12}: {hyde_metrics[k]:.4f}')

=== Contriever + HyDE ===
Recall@1    : 0.4334
Recall@10   : 0.7868
Recall@100  : 0.9453
MRR@10      : 0.5662
nDCG@10     : 0.6111


In [39]:
out_hyde = Path('results/contriever_hyde')
out_hyde.mkdir(parents=True, exist_ok=True)

payload = {
    'config': {
        'mode': 'contriever_hyde',
        'embed_model': CONTRIEVER_MODEL,
        'llm_model': 'gpt-4o-mini',
        'split': SPLIT,
        'data_source': data_source,
        'top_ks': TOP_KS_LOCAL,
        'max_queries': MAX_QUERIES_LOCAL,
    },
    'contriever_hyde': hyde_metrics,
}
with (out_hyde / 'metrics.json').open('w') as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)
with (out_hyde / 'per_query_results.jsonl').open('w') as f:
    for row in hyde_rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

# Also save the generated hyde documents for reference
with (out_hyde / 'hyde_docs.jsonl').open('w') as f:
    for qid in query_ids:
        f.write(json.dumps({'query_id': qid, 'query': queries[qid], 'hyde_doc': hyde_docs[qid]}, ensure_ascii=False) + '\n')

print(f'Saved to {out_hyde}')

Saved to results/contriever_hyde


## Full Comparison Table

Reads saved `metrics.json` from all experiment folders.

In [41]:
def load_metrics(path: Path, key: str):
    if not path.exists():
        return None
    with path.open() as f:
        return json.load(f).get(key)

root = Path('results')
experiments = [
    ('BM25',            load_metrics(root / 'bm25'      / 'metrics.json', 'bm25')),
    ('BGE-Baseline',    load_metrics(root / 'baseline'  / 'metrics.json', 'baseline')),
    ('BGE-HyDE',        load_metrics(root / 'hyde'      / 'metrics.json', 'hyde')),
    ('Contriever-Base', baseline_metrics),
    ('Contriever-HyDE', hyde_metrics),
]

keys = ['Recall@1', 'Recall@10', 'Recall@100', 'MRR@10', 'nDCG@10']
col  = 12
header = f"{'Method':<18}" + ''.join(f'{k:>{col}}' for k in keys)
print(header)
print('-' * len(header))
for name, m in experiments:
    if m is None:
        print(f'{name:<18}  (results not found)')
    else:
        print(f"{name:<18}" + ''.join(f"{m.get(k, float('nan')):>{col}.4f}" for k in keys))

Method                Recall@1   Recall@10  Recall@100      MRR@10     nDCG@10
------------------------------------------------------------------------------
BM25                    0.5371      0.8072      0.9253      0.6460      0.6799
BGE-Baseline            0.5757      0.8566      0.9633      0.6936      0.7296
BGE-HyDE                0.5683      0.8824      0.9667      0.6901      0.7338
Contriever-Base         0.4183      0.7353      0.9027      0.5453      0.5847
Contriever-HyDE         0.4334      0.7868      0.9453      0.5662      0.6111
